# Computer Assignment Template: Belief Propagation (BP) for Sudoku

This notebook is a template for implementing a basic loopy sum-product (belief propagation) decoder for Sudoku.

**Constraints for this assignment version**
- Use only **pairwise not-equal** constraints (complete graph within each row/column/3×3 box).
- Implement **probability-domain** BP.
- Run a fixed number of iterations (e.g., **50**) with damping (e.g., **α = 0.5**).
- Minimal extra features: no guided decimation, no backtracking, no command line interface, no dataset harness.

Complete all `TODO` sections below.


## Representation

- Sudoku variables: 81 cells, indexed by `i = 9*r + c`.
- Alphabet: digits `1..9`, represented as indices `0..8` in arrays.
- Unary factors (clues) represented as `phi[i, d]` for digit `d` (0..8).
- Pairwise constraints: for each edge `(i,j)` we have factor `f_neq(x_i,x_j)=1[x_i ≠ x_j]`.

### Messages

Store **directed** messages on each neighbor edge:

- `m[i, k, d]` denotes the message from variable `i` to its `k`-th neighbor `j = neigh[i,k]`, evaluated at digit `d`.

For a not-equal factor, the factor-to-variable update is:
$$
\hat m_{ij\to i}(d) = 1 - m_{j\to ij}(d)
$$
assuming the incoming message is normalized (sums to 1).

Then the direct variable-to-neighbor update is:
$$
m_{j\to i}'(d) \propto f_i(d)\prod_{k\in N(i)\setminus\{j\}} \hat{m}_{ik\to i}(d) = f_i(d)\prod_{k\in N(i)\setminus\{j\}} (1- m_{i \to ik\to i}(d).
$$
Normalize output to sum to 1 over digits, and apply damping:
$$
m^{(t+1)}=(1-\alpha)m^{(t)}+\alpha m'.
$$


In [ ]:
import numpy as np

EPS = 1e-12  # small number to avoid division by zero

def parse_grid(lines_or_string):
    # Parse a Sudoku puzzle:
    # - either an 81-character string, or
    # - a list/tuple of 9 strings of length 9
    if isinstance(lines_or_string, (list, tuple)):
        s = "".join(lines_or_string)
    else:
        s = str(lines_or_string)
    s = "".join(ch for ch in s if ch not in " \n\t\r")
    if len(s) != 81:
        raise ValueError("Need exactly 81 characters (9 lines of 9 or one 81-char string).")
    grid = np.zeros((9, 9), dtype=np.int32)
    for i, ch in enumerate(s):
        r, c = divmod(i, 9)
        if ch in ".0":
            grid[r, c] = 0
        elif ch.isdigit() and ch != "0":
            grid[r, c] = int(ch)
        else:
            raise ValueError(f"Invalid character {ch!r} at position {i}.")
    return grid

def pretty(grid):
    # Pretty-print a 9-by-9 grid (0 rendered as '.')
    out = []
    for r in range(9):
        if r % 3 == 0 and r != 0:
            out.append("-" * 21)
        row = []
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row.append("|")
            v = int(grid[r, c])
            row.append(str(v) if v != 0 else ".")
        out.append(" ".join(row))
    return "\n".join(out)

# Test example from the assignment handout
puzzle = [
    "003020600",
    "900305001",
    "001806400",
    "008102900",
    "700000008",
    "006708200",
    "002609500",
    "800203009",
    "005010300",
]
grid = parse_grid(puzzle)
print(pretty(grid))


## Step 1: Build the Sudoku graph (neighbors)

Two cells are neighbors if they share the same row, column, or 3×3 box.

Each cell should have exactly **20** distinct neighbors.

Implement `build_neighbors()` returning:
- `neigh` : shape `(81,20)` with neighbor indices for each variable
- `rev`   : shape `(81,20)` reverse indices to find the message in the opposite direction

`rev[i,k]` is the index `k2` such that `neigh[ neigh[i,k], k2 ] == i`.

This mapping lets you retrieve the reverse message in O(1) time.


In [ ]:
def build_neighbors():
    '''
    Returns
    -------
    neigh : (81, 20) int array
        neigh[i,k] is the k-th neighbor j of i.
    rev : (81, 20) int array
        rev[i,k] is the index k2 such that neigh[ neigh[i,k], k2 ] == i.

    TODO: Implement.
    Hints:
    - Use sets to accumulate neighbors, then sort for a deterministic order.
    - After building neigh, build rev by scanning neighbor lists.
    - Sanity check: each cell has exactly 20 neighbors.
    '''
    # TODO: replace this placeholder
    raise NotImplementedError("build_neighbors() not yet implemented.")

# Uncomment when implemented:
# neigh, rev = build_neighbors()
# print(neigh.shape, rev.shape)
# print("Neighbors of cell 0:", neigh[0])


## Step 2: Unary factors from givens

Create `phi[i,d]` where:
- If cell `i` is a given digit `v`, then `phi[i,:]` is one-hot at `v-1`.
- If cell `i` is blank, then `phi[i,d]=1` for all digits consistent with the givens (you may prune digits already used in the same row/col/box).

If a contradiction is present (e.g., a blank has no allowed digit), return `None`.


In [ ]:
def unary_from_givens(grid):
    '''
    Parameters
    ----------
    grid : (9,9) int array

    Returns
    -------
    phi : (81,9) float array or None
        Unary compatibility over digits 0..8.
        Return None if a contradiction is detected.
    '''
    # TODO: implement
    raise NotImplementedError("unary_from_givens() not yet implemented.")


## Step 3: Core BP update

Implement `bp_sudoku(phi, neigh, rev, iters=50, damping=0.5)`.

- Initialize each outgoing message from variable `i` proportional to `phi[i,:]`.
- Iterate:
  - for each directed edge `i -> j`, compute
    `m_new[i->j](d) ∝ phi[i](d) * ∏_{k in N(i)\{j}} (1 - m[k->i](d))`
  - normalize messages to sum to 1 over digits
  - apply damping
- After iterations, compute beliefs
  `b[i,d] ∝ phi[i,d] * ∏_{k in N(i)} (1 - m[k->i](d))`
  and normalize over digits.

Numerical handling:
- If an outgoing message becomes all zeros, replace it by a uniform distribution over allowed digits (where `phi[i,d] > 0`).


In [ ]:
def bp_sudoku(phi, neigh, rev, iters=50, damping=0.5):
    '''
    Loopy BP on the pairwise not-equal Sudoku graph.

    Parameters
    ----------
    phi : (81,9) float array
        Unary factors.
    neigh, rev : (81,20) int arrays
        Neighbor list and reverse index mapping.
    iters : int
        Number of BP iterations.
    damping : float in [0,1]
        Damping parameter alpha.

    Returns
    -------
    m : (81,20,9) float array
        Directed messages m[i,k,d] = message from i to neigh[i,k] at digit d.
    beliefs : (81,9) float array
        Approximate marginal beliefs.
    '''
    nvars, deg = neigh.shape
    q = phi.shape[1]
    assert nvars == 81 and deg == 20 and q == 9

    # -----------------------
    # TODO (Initialization):
    # -----------------------
    # Initialize m so that for each variable i and each neighbor k:
    #   m[i,k,:] = normalized(phi[i,:])
    m = np.zeros((nvars, deg, q), dtype=np.float64)
    raise NotImplementedError("Initialize messages m.")

    # --------------
    # BP iterations
    # --------------
    for t in range(iters):
        m_new = np.zeros_like(m)

        # -----------------------
        # TODO (Message updates):
        # -----------------------
        # For each variable i, for each neighbor slot k (target j):
        #   incoming from neighbor u to i at digit d is (1 - m[u, rev_i_u, d])
        #   where u = neigh[i, k2] and rev_i_u = rev[i, k2]
        #
        # Use product over neighbors excluding the target edge.

        # ----------------------
        # TODO (Normalization):
        # ----------------------
        # Normalize each m_new[i,k,:] to sum to 1; if sum=0, fall back.

        # ---------------
        # TODO (Damping):
        # ---------------
        # m = (1-damping)*m + damping*m_new
        # renormalize if needed

        raise NotImplementedError("Finish BP iteration loop.")

    # -----------------
    # TODO (Beliefs):
    # -----------------
    # beliefs[i,d] ∝ phi[i,d] * ∏_{k in N(i)} (1 - m[neigh->i](d))
    beliefs = np.zeros((nvars, q), dtype=np.float64)
    raise NotImplementedError("Compute beliefs.")

    return m, beliefs


## Step 4: Read out a solution and validate

Once you have beliefs `b[i,d]`, read out:
$$ \hat x_i = \arg\max_d b[i,d] $$

Then check:
- respects givens
- each row/col/box contains digits 1..9 exactly once

Implement `is_valid_solution`.


In [ ]:
def beliefs_to_grid(beliefs, givens_grid):
    sol = givens_grid.copy()
    for i in range(81):
        r, c = divmod(i, 9)
        if sol[r, c] != 0:
            continue
        sol[r, c] = int(np.argmax(beliefs[i, :])) + 1
    return sol

def is_valid_solution(sol, givens_grid):
    '''
    Return True iff sol is a valid Sudoku solution that respects givens_grid.
    TODO: implement.
    '''
    raise NotImplementedError("is_valid_solution() not yet implemented.")


## Step 5: Early termination (optional)

During BP iterations, you may periodically:
1. Form a best-guess grid from current beliefs.
2. If it is a valid Sudoku solution, stop early.

You may also detect contradictions (e.g., a variable has no allowed digits after pruning) and halt.


## Run the solver on the provided example

After implementing the TODOs, this cell should run end-to-end.


In [ ]:
grid = parse_grid(puzzle)
phi = unary_from_givens(grid)
assert phi is not None, "Contradiction in givens."

neigh, rev = build_neighbors()

m, beliefs = bp_sudoku(phi, neigh, rev, iters=50, damping=0.5)
sol = beliefs_to_grid(beliefs, grid)

print("BP best guess:")
print(pretty(sol))
print("Valid Sudoku solution?", is_valid_solution(sol, grid))


## What to hand in

1. Your completed BP solver in this notebook (or exported `.py`) that solves at least the provided example.
2. Written responses (by hand) addressing the theory questions in the assignment prompt (factorization, constraint counting, update derivations, early termination, etc.).
3. (Optional/extra) A second “GenAI-enhanced” version with additional features, with a short report describing what you added.
